# 02_03 - Peak Labeling and Evaluation Metrics

Two more building blocks before any model gets trained: the peak-risk labels the classifier will learn to predict, and the evaluation functions both models will be scored with.

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parents[1] / "src"))

import pandas as pd
from common.folds import iter_folds
from common.labeling import compute_peak_thresholds, label_peaks
from common.metrics import (
    forecast_metrics,
    forecast_metrics_by_horizon,
    peak_risk_metrics,
    top_k_capture_rate,
    calibration_data,
)

DATA_PATH = Path.cwd().parents[1] / "data" / "interim" / "fsa_hourly_master.parquet"
df = pd.read_parquet(DATA_PATH)

## 1. Peak labeling

An hour counts as a peak if its consumption exceeds the 97.5th percentile for that FSA and season, and that threshold is computed only from each fold's own training data, then applied to both train and test of that fold. All 3 folds are checked together below.

In [2]:
rows = []
for fold_name, train_df, test_df in iter_folds(df):
    thresholds = compute_peak_thresholds(train_df)
    train_labeled = label_peaks(train_df, thresholds)
    test_labeled = label_peaks(test_df, thresholds)
    rows.append({
        "fold": fold_name,
        "n_thresholds": len(thresholds),
        "train_pct_peak": round(train_labeled["actual_peak"].mean() * 100, 2),
        "test_pct_peak": round(test_labeled["actual_peak"].mean() * 100, 2),
    })

pd.DataFrame(rows)

,fold,n_thresholds,train_pct_peak,test_pct_peak
0,fold_1,24,2.51,10.95
1,fold_2,24,2.50,4.00
2,fold_3,24,2.50,6.93


24 thresholds per fold (6 FSAs x 4 seasons), and train always lands almost exactly on 2.5% peak, expected, since the threshold is by definition the 97.5th percentile of that same train data.

Test does not land anywhere near 2.5%, though: fold_1 (test year 2023) comes out at close to 11%, fold_2 (2024) around 4%, fold_3 (2025) around 7%. That is not a bug: the threshold is fixed from earlier, smaller years, and if consumption trends upward over time (which lines up with the step increases already seen in `PREMISE_COUNT`), a growing share of hours in the test year end up crossing a threshold calibrated on the past. Worth keeping in mind for the classifier later, the real class imbalance at evaluation time is milder and more fold-dependent than the 97.5/2.5 split the training data always shows.

## 2. Forecast metrics

There is no trained model yet, so these are checked against small hand-picked numbers instead, values chosen so the correct answer can be worked out by hand first, then compared to what the function returns.

`actual = [10, 20, 30]`, `prediction = [12, 18, 33]`: signed errors (actual - prediction) are -2, 2, -3.
- MAE = (2+2+3)/3 = 2.333
- WAPE = 7/60 = 0.1167
- RMSE = sqrt((4+4+9)/3) = 2.380
- MAPE = mean(2/10, 2/20, 3/30) x 100 = 13.33%
- Bias = (-2+2-3)/3 = -1.0 (negative, the model over-predicts by 1 kWh on average here, unlike the other four this one keeps the sign instead of taking an absolute value)

In [3]:
toy_actual = [10, 20, 30]
toy_prediction = [12, 18, 33]

forecast_metrics(toy_actual, toy_prediction)

{'mae': 2.3333333333333335,
 'wape': 0.11666666666666667,
 'rmse': 2.3804761428476167,
 'mape': 13.333333333333334,
 'bias': -1.0}

All four match the hand-calculated values above. Now checking that the by-horizon breakdown computes each horizon's metrics independently, using two tiny horizon groups with known errors (horizon 1: errors of 2; horizon 2: errors of 5).

In [4]:
toy_df = pd.DataFrame({
    "horizon": [1, 1, 2, 2],
    "actual": [10, 20, 10, 20],
    "prediction": [12, 18, 15, 25],
})

forecast_metrics_by_horizon(toy_df)

,mae,wape,rmse,mape,bias
horizon,,,,,
1,2.0,0.133333,2.0,15.0,0.0
2,5.0,0.333333,5.0,37.5,-5.0


## 3. Peak-risk metrics

Same idea, with a toy example where the prediction is deliberately perfect: `actual_peak = [0, 0, 1, 1]`, `peak_probability = [0.1, 0.4, 0.6, 0.9]` (correctly ranks the two peaks highest), `predicted_peak = [0, 0, 1, 1]` (matches exactly). Since the classification is perfect, precision/recall/f1/PR-AUC/ROC-AUC should all come out to 1.0, and peak_rate_bias should come out to 0.0, since the predicted peak rate (2/4) exactly matches the actual peak rate (2/4).

In [5]:
toy_actual_peak = [0, 0, 1, 1]
toy_probability = [0.1, 0.4, 0.6, 0.9]
toy_predicted_peak = [0, 0, 1, 1]

peak_risk_metrics(toy_actual_peak, toy_probability, toy_predicted_peak)

{'pr_auc': 1.0,
 'precision': 1.0,
 'recall': 1.0,
 'f1': 1.0,
 'roc_auc': 1.0,
 'brier': 0.08500000000000002,
 'peak_rate_bias': 0.0}

Brier score comes out to 0.085 (mean of the four squared errors between probability and actual label: 0.01, 0.16, 0.16, 0.01), lower is better, not 0 here because the probabilities are not exactly 0 or 1, even though the classification itself is perfect. `peak_rate_bias` comes out to 0.0 as expected, confirming it correctly cancels out when the predicted and actual peak counts match, this toy example cannot tell whether it also cancels out for the wrong reason (predicting the right count of peaks but flagging the wrong hours), that distinction only matters once real predictions are checked.

`top_k_capture_rate` with k=2 on the same toy data: the two highest-probability rows (indices 2 and 3, probabilities 0.6 and 0.9) are exactly the two actual peaks, so all of them get captured in the top 2, the rate should be 1.0.

In [6]:
top_k_capture_rate(toy_actual_peak, toy_probability, k=2)

1.0

And `calibration_data` splits the 4 toy points into 2 equal-sized bins by probability, the low bin (0.1, 0.4) has an average predicted probability of 0.25 and an observed peak frequency of 0 (neither point is a real peak), the high bin (0.6, 0.9) averages 0.75 predicted with an observed frequency of 1 (both points are real peaks). Real predictions from a trained model will not line up this cleanly, but the mechanics check out.

In [7]:
calibration_data(toy_actual_peak, toy_probability, n_bins=2)

,predicted_probability,observed_frequency
0,0.25,0.0
1,0.75,1.0
